# 01 - Rebuild Chroma DB dengan Acronym-Expanded Text

**Tujuan**: re-embed semua 1.706 chunks dengan text yang sudah di-expand acronym-nya,
lalu simpan ke ChromaDB baru di `notebooks/pubmedqa_chroma_expanded`.

**Estimasi**:
- Cost API: ~$0.005 (1706 chunks x 150 tokens avg x $0.02/1M)
- Waktu: ~5 menit (batch 100)

**Input**: `notebooks/pubmedqa_bm25_expanded.pkl` (sudah dibuat oleh build_expanded_index.py)

**Output**: `notebooks/pubmedqa_chroma_expanded/` (folder ChromaDB baru)


In [1]:
import os, sys, time, pickle
from pathlib import Path
from dataclasses import dataclass

# Set OpenAI API key (Anda bisa juga set via env var)
# # os.environ["OPENAI_API_KEY"] = "<REDACTED — set via shell env or .env file>"

from openai import OpenAI
import chromadb

# ============================================================
# Document class (compat dengan pickle dari main notebook)
# ============================================================
@dataclass
class Document:
    text: str; pubid: str; question: str
    section_label: str; answer: str; decision: str

import __main__
__main__.Document = Document

# ============================================================
# Paths
# ============================================================
HERE = Path(".").resolve()
NOTEBOOKS_DIR = HERE.parent if HERE.name == "BM25 Expansion" else HERE
BM25_EXPANDED_PATH = NOTEBOOKS_DIR / "pubmedqa_bm25_expanded.pkl"
CHROMA_EXPANDED_PATH = NOTEBOOKS_DIR / "pubmedqa_chroma_expanded"

print(f"BM25 expanded path : {BM25_EXPANDED_PATH} (exists: {BM25_EXPANDED_PATH.exists()})")
print(f"Chroma target path : {CHROMA_EXPANDED_PATH}")
print(f"Will create new collection \"pubmedqa_docs_expanded\"")


BM25 expanded path : C:\Users\Ricky Wijaya\Documents\STI\Semester 8\TA\Code TA\notebooks\pubmedqa_bm25_expanded.pkl (exists: True)
Chroma target path : C:\Users\Ricky Wijaya\Documents\STI\Semester 8\TA\Code TA\notebooks\pubmedqa_chroma_expanded
Will create new collection "pubmedqa_docs_expanded"


In [5]:
import os

# Set env var
# os.environ["OPENAI_API_KEY"] = "<REDACTED — set via shell env or .env file>"

In [6]:
# Load expanded BM25 to get all 1706 expanded chunks
with open(BM25_EXPANDED_PATH, "rb") as f:
    saved = pickle.load(f)
documents = saved["documents"]
print(f"Loaded {len(documents)} expanded chunks")
print()
print("Sample expanded chunk (paper 11729377, METHODS section):")
for d in documents:
    if d.pubid == "11729377" and d.section_label == "METHODS":
        print(d.text[:300])
        break


Loaded 1706 expanded chunks

Sample expanded chunk (paper 11729377, METHODS section):
Outcomes and postoperative liver function of 43 primary LRT (living-related liver transplantation) patients were compared with those of 49 primary SLT (split-liver transplantation) patients (14 ex situ, 35 in situ) with known graft weight performed between April 1996 and December 2000. Survival rate


In [7]:
# Setup OpenAI client
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY belum di-set. Set via env var atau os.environ.")

client = OpenAI(api_key=api_key)
EMBED_MODEL = "text-embedding-3-small"  # 1536 dim

# Smoke test
_emb = client.embeddings.create(model=EMBED_MODEL, input=["aspirin reduces heart attack risk"])
print(f"Embedding dim: {len(_emb.data[0].embedding)}  (expected 1536)")


Embedding dim: 1536  (expected 1536)


In [8]:
# ============================================================
# Build Chroma collection dengan embedding expanded chunks
# ============================================================
def openai_embed(texts, model=EMBED_MODEL, max_retries=5):
    """Wrapper dengan retry."""
    for attempt in range(max_retries):
        try:
            resp = client.embeddings.create(model=model, input=texts)
            return [d.embedding for d in resp.data]
        except Exception as e:
            err = str(e)
            if "429" in err or "rate" in err.lower():
                wait = (attempt + 1) * 10
                print(f"  [Rate limit] Tunggu {wait}s...")
                time.sleep(wait)
            elif "500" in err or "502" in err or "503" in err:
                time.sleep((attempt + 1) * 5)
            else:
                raise
    raise RuntimeError("OpenAI embeddings gagal setelah 5 retry.")


# Initialize Chroma client
chroma_client = chromadb.PersistentClient(path=str(CHROMA_EXPANDED_PATH))
COLL_NAME = "pubmedqa_docs_expanded"

# Hapus kalau sudah ada (rebuild bersih)
try:
    chroma_client.delete_collection(name=COLL_NAME)
    print(f"Deleted existing collection \"{COLL_NAME}\"")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLL_NAME,
    metadata={"hnsw:space": "cosine"}
)
print(f"Created new collection \"{COLL_NAME}\"")

# Batch embed
BATCH = 100
t0 = time.time()
total = len(documents)

for start in range(0, total, BATCH):
    batch_docs = documents[start:start + BATCH]
    batch_texts = [d.text[:8000] for d in batch_docs]  # truncate untuk safety
    batch_ids = [str(start + i) for i in range(len(batch_docs))]
    batch_meta = [{"pubid": d.pubid, "section": d.section_label} for d in batch_docs]

    embeddings = openai_embed(batch_texts)
    collection.add(
        ids=batch_ids,
        documents=batch_texts,
        metadatas=batch_meta,
        embeddings=embeddings,
    )
    done = start + len(batch_docs)
    elapsed = time.time() - t0
    eta = elapsed / done * (total - done) / 60 if done < total else 0
    print(f"  [{done}/{total}] embedded | {elapsed:.0f}s elapsed | ETA {eta:.1f} mnt")

print(f"\nSelesai dalam {(time.time()-t0)/60:.1f} menit")
print(f"Total chunks di Chroma: {collection.count()}")


Created new collection "pubmedqa_docs_expanded"
  [100/1706] embedded | 2s elapsed | ETA 0.4 mnt
  [200/1706] embedded | 4s elapsed | ETA 0.4 mnt
  [300/1706] embedded | 5s elapsed | ETA 0.4 mnt
  [400/1706] embedded | 5s elapsed | ETA 0.3 mnt
  [500/1706] embedded | 7s elapsed | ETA 0.3 mnt
  [600/1706] embedded | 8s elapsed | ETA 0.2 mnt
  [700/1706] embedded | 9s elapsed | ETA 0.2 mnt
  [800/1706] embedded | 11s elapsed | ETA 0.2 mnt
  [900/1706] embedded | 12s elapsed | ETA 0.2 mnt
  [1000/1706] embedded | 13s elapsed | ETA 0.2 mnt
  [1100/1706] embedded | 14s elapsed | ETA 0.1 mnt
  [1200/1706] embedded | 15s elapsed | ETA 0.1 mnt
  [1300/1706] embedded | 17s elapsed | ETA 0.1 mnt
  [1400/1706] embedded | 19s elapsed | ETA 0.1 mnt
  [1500/1706] embedded | 21s elapsed | ETA 0.0 mnt
  [1600/1706] embedded | 22s elapsed | ETA 0.0 mnt
  [1700/1706] embedded | 24s elapsed | ETA 0.0 mnt
  [1706/1706] embedded | 24s elapsed | ETA 0.0 mnt

Selesai dalam 0.4 menit
Total chunks di Chroma: 1

In [9]:
# ============================================================
# Verifikasi: query test untuk paper LRT/SLT (idx 16)
# ============================================================
test_query = "Is there still a need for living-related liver transplantation in children?"

qvec = openai_embed([test_query])[0]
results = collection.query(
    query_embeddings=[qvec],
    n_results=10,
    include=["distances", "metadatas"]
)

print(f"Query: {test_query}")
print(f"\nDense top-10 (Chroma EXPANDED):")
print(f"{'rank':>4} {'doc_id':>7} {'pubid':>10} {'section':<28} {'sim':>6}")
for rank, (did, dist, meta) in enumerate(zip(results["ids"][0], results["distances"][0], results["metadatas"][0]), 1):
    sim = 1 - dist
    src = "*SOURCE*" if meta["pubid"] == "11729377" else ""
    print(f"{rank:>4} {did:>7} {meta['pubid']:>10} {meta['section']:<28} {sim:.3f} {src}")


Query: Is there still a need for living-related liver transplantation in children?

Dense top-10 (Chroma EXPANDED):
rank  doc_id      pubid section                         sim
   1      51   11729377 METHODS                      0.589 *SOURCE*
   2      50   11729377 SUMMARY BACKGROUND DATA      0.583 *SOURCE*
   3      52   11729377 RESULTS                      0.581 *SOURCE*
   4      49   11729377 OBJECTIVE                    0.575 *SOURCE*
   5     157   20571467 RESULTS                      0.500 
   6     155   20571467 BACKGROUND                   0.485 
   7    1231   20530150 RESULTS                      0.479 
   8     156   20571467 METHODS                      0.479 
   9    1230   20530150 METHODS                      0.473 
  10     990   21850494 RESULTS                      0.473 


## Verifikasi sukses

Kalau output di atas menunjukkan **METHODS dan RESULTS dari paper sumber (11729377)**
masuk top-10, artinya re-embed berhasil dan dense retrieval sekarang juga terbantu
oleh acronym expansion.

**Lanjut ke notebook 02** untuk run full baseline OpenAI dengan kedua index expanded.
